# Demo 2 Visual data processing

In this notebook, we add visual data to our chroma database and explore the capabilities and limitations of this method. Note that you should complete *"1_web_scraper.ipynb“* and *”2_embedding_generation.ipynb“* before running this notebook. 

### Approach

We will:

1. Collect data (= take photo)​.
2. Generate AI description (spatial layout, colors, objects, etc.)​.
3. Generate embeddings and save to vector database.
4. Run inference on different inquiries to test its limit.

### Goal

Find out if the chatbot can answer the following question:

*What is the room number of Thematic Workshop A?*


## Install dependencies and Import libraries
Install the libraries needed for OpenAI, ChromaDB, and image encoding.

In [7]:
!pip install pybase64 openai chromadb

In [20]:
from dotenv import load_dotenv
import os
import chromadb
import os
from openai import AzureOpenAI
import base64
from pathlib import Path
import chromadb
BASE_DIR = Path.cwd().parent

## Initialize the vision client
Load environment variables and create the Azure OpenAI client for image understanding.

In [ ]:
load_dotenv(BASE_DIR / ".env")

API_Key = os.getenv("AZURE_OPENAI_API_KEY")

if not API_Key:
    raise RuntimeError("Missing Azure OpenAI credentials. Set AZURE_OPENAI_API_KEY in .env or environment.")

client = AzureOpenAI(
    azure_endpoint="https://api-iw.azure-api.net/sig-shared-jpeast/deployments/gpt-4o-mini/chat/completions?api-version=2025-01-01-preview",
    api_key=API_Key,
    api_version="2025-01-01-preview",
)

## Describe the image
Load the image file, encode it to base64, and ask the vision model for a detailed description.

In [4]:
# Path to your image
image_path = "makerspace_A.jpeg"
# Read image and encode as base64
with open(image_path, "rb") as image_file:
    base64_image = base64.b64encode(image_file.read()).decode("utf-8")

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "Describe this image in detail."
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{base64_image}"
                }
            }
        ]
    }
]

description = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages
).choices[0].message.content

## Print the image description
Format and display the model-generated description for easy reading.

In [5]:
import textwrap

print("Image Description:\n")
for paragraph in description.split("\n\n"):
    print(textwrap.fill(paragraph, width=90))
    print()

Image Description:

The image depicts a spacious, modern classroom or workshop setting with sleek,
minimalistic design elements. The room features a high ceiling with exposed ductwork and
numerous bright, rectangular LED lights providing ample illumination.

In the center, there are several large, light wooden tables arranged in a collaborative
setup. Each table is surrounded by black stools, creating an inviting environment for
group work or discussions. On a few tables, there are scattered papers and tools,
indicating a practical, hands-on focus.

To one side of the room, there is a large screen displaying information, possibly related
to ongoing projects or presentations, and a smaller monitor nearby. These devices
facilitate teaching and collaboration.

The walls have large windows, allowing natural light to filter in and showcasing green
plants on the windowsills, which add a touch of nature to the industrial aesthetic.
Overall, the space is designed to foster creativity and innov

## Prepare the vector database
Connect to ChromaDB, delete the existing collection if present, and create a fresh collection for storing embeddings.

In [6]:
# Load .env from parent folder
load_dotenv(BASE_DIR / ".env")
CHROMA_PATH = os.getenv("CHROMA_PATH") or "chroma_db/chroma_db"

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

COLLECTION_NAME = "Innowing_db"

try:
    chroma_client.delete_collection(name=COLLECTION_NAME)
    print(f"🗑️  Deleted existing collection '{COLLECTION_NAME}' for fresh ingest.")
except:
    pass

collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)

## Set embedding model
Choose the embedding model and initialize the second Azure OpenAI client for vector generation.

In [9]:

EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL") or "text-embedding-3-small"

API_Key_2 = os.getenv("AZURE_OPENAI_API_KEY_2")
if not API_Key_2:
    raise RuntimeError("Missing Azure OpenAI credentials.")

client_2 = AzureOpenAI(
    azure_endpoint="https://api-iw.azure-api.net/sig-embedding/openai/deployments/text-embedding-3-small/embeddings?api-version=2024-10-21",
    api_key=API_Key_2,
    api_version="2024-10-21",
)


## Embed and store the description
Generate an embedding for the image description and insert it into ChromaDB.

In [ ]:
def get_embedding(text: str) -> List[float]:
    response = client_2.embeddings.create(
        input=[text.replace("\n", " ")],
        model=EMBEDDING_MODEL,
    )
    return response.data[0].embedding

print("🧬 Generating embeddings and storing in ChromaDB (this may take a while)...")

BATCH_SIZE = 50

embedding = get_embedding(description)

description_id = "image_1"

collection.upsert(
        embeddings=[embedding],
        documents=[description],
        ids=[description_id]
    )

print("\n🎉 Ingestion complete! Description added to ChromaDB collection.")

🧬 Generating embeddings and storing in ChromaDB (this may take a while)...

🎉 Ingestion complete! Description added to ChromaDB collection.


## Define a query
Set the user question that will be sent to the retrieval and LLM pipeline.

In [11]:
question = "How does makerspace A look like?"

## Retrieve similar documents
Create an embedding for the query and search the ChromaDB collection for matching text chunks.

In [13]:
top_k = 5

if not question.strip():
    results = collection.peek(limit=top_k)
    context_docs = [{"text": doc, "url": meta.get("url", ""), "score": 1.0}
            for doc, meta in zip(results["documents"], results["metadatas"])]

response = client.embeddings.create(
    input=question.replace("\n", " "),
    model=os.getenv("EMBEDDING_MODEL", "text-embedding-3-small"),
)
query_embedding = response.data[0].embedding

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=top_k,
    include=["documents", "metadatas", "distances"]
)

context_docs = []
for doc, meta, distance in zip(results["documents"][0], results["metadatas"][0], results["distances"][0]):
    # Convert distance to similarity score (Chroma uses cosine distance by default)
    score = 1 - distance
    context_docs.append({
        "text": doc,
        "score": round(score, 4)
    })

## Display retrieved passages
Show the top matching documents so you can inspect what the retriever found.

In [14]:
print("Retrieved Context Documents:")
for idx, doc in enumerate(context_docs, 1):
    print(f"{idx}. {doc['text'][:100]}... (Score: {doc['score']})")

Retrieved Context Documents:
1. The image depicts a spacious, modern classroom or workshop setting with sleek, minimalistic design e... (Score: 0.3457)


## Build prompt context
Combine the retrieved documents into a single context block for the LLM.

In [16]:
context_parts = []
for i, doc in enumerate(context_docs, 1):
        context_parts.append(
            f"Document {i}:\n{doc['text']}"
        )

context = "\n\n---\n\n".join(context_parts)

print("\nConstructed Context for LLM:")
print(context)


Constructed Context for LLM:
Document 1:
The image depicts a spacious, modern classroom or workshop setting with sleek, minimalistic design elements. The room features a high ceiling with exposed ductwork and numerous bright, rectangular LED lights providing ample illumination.

In the center, there are several large, light wooden tables arranged in a collaborative setup. Each table is surrounded by black stools, creating an inviting environment for group work or discussions. On a few tables, there are scattered papers and tools, indicating a practical, hands-on focus.

To one side of the room, there is a large screen displaying information, possibly related to ongoing projects or presentations, and a smaller monitor nearby. These devices facilitate teaching and collaboration.

The walls have large windows, allowing natural light to filter in and showcasing green plants on the windowsills, which add a touch of nature to the industrial aesthetic. Overall, the space is designed to foste

## Generate the final answer
Send the retrieved context and the question to the chat model for a grounded response.

In [ ]:
messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant for HKU InnoWings and InnoAcademy. "
                "Answer the question using ONLY the provided context documents. "
                "If the answer cannot be found in the context, say: "
                "'I don't have enough information based on the provided documents.' "
                "Be concise, accurate, and professional. Always cite the source URL when possible."
            )
        },
        {
            "role": "user",
            "content": f"Context documents:\n{context}\n\nQuestion: {question}"
        }
    ]

answer = client.chat.completions.create(
        #model="gpt-4o-mini",
        model="gpt-5-mini"
        messages=messages
    ).choices[0].message.content

print(answer)

The image depicts a spacious, modern classroom or workshop setting with a sleek, minimalistic design. It features a high ceiling with exposed ductwork and bright, rectangular LED lights. In the center, there are large, light wooden tables arranged collaboratively, surrounded by black stools. Scattered papers and tools suggest a practical focus. A large screen displays information, with a smaller monitor nearby for teaching and collaboration. The walls have large windows allowing natural light in and showcasing green plants on the windowsills, contributing to the inviting atmosphere designed to foster creativity and innovation.


# Can the AI count?
Ask the vision model a counting question about the same image to test its visual reasoning.

In [ ]:
# Path to your image
image_path = "makerspace_A.jpeg"
# Read image and encode as base64
with open(image_path, "rb") as image_file:
    base64_image = base64.b64encode(image_file.read()).decode("utf-8")

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "How many desks are there in this image?"
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:image/jpeg;base64,{base64_image}"
                }
            }
        ]
    }
]

answer = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=messages
).choices[0].message.content

import textwrap

print("Image Description:\n")
for paragraph in answer.split("\n\n"):
    print(textwrap.fill(paragraph, width=90))
    print()

Image Description:
There are ten desks in the image.
